# Neural Networks 
- Architecture 
- Regularization
- Hyperparameter Tuning

In this notebook we explore two core questions:

1. How does architecture and regularization shape what a neural network learns?
2. How do we choose hyperparameters rigorously instead of by trial and error?

Dataset: `make_moons` — a non-linearly separable 2-class problem, ideal for visualizing decision boundaries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 44
COLORS = ["#4C72B0", "#DD8452"]  # blue / orange

## Dataset

We generate samples from `make_moons` with moderate noise. The two crescents overlap slightly, creating a realistic classification challenge.

In [ ]:
X, y = make_moons(n_samples=200, noise=0.25, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for cls, label in enumerate(["Class 0", "Class 1"]):
    mask = y == cls
    ax.scatter(X[mask, 0], X[mask, 1], c=COLORS[cls], label=label,
               edgecolors="k", linewidths=0.3, s=40, alpha=0.8)
ax.set_title("make_moons dataset (noise=0.25)")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend()
plt.tight_layout()
plt.show()

## Part 1: How Does Architecture and Regularization Affect the Model?

We train three `MLPClassifier` models and compare their decision boundaries:

| Model | Architecture | Regularization | Expected behavior |
|---|---|---|---|
| Small | (5,) | none | Underfitting — boundary too simple |
| Large | (100, 100, 100) | none | Overfitting — boundary too complex |
| Large + Reg | (100, 100, 100) | alpha=1.0 | Regularized — boundary generalizes |

In [ ]:
def plot_decision_boundary(ax, model, X_train, y_train, title,
                           X_test=None, y_test=None):
    h = 0.02
    X_all = X_train if X_test is None else np.vstack([X_train, X_test])
    x_min, x_max = X_all[:, 0].min() - 0.5, X_all[:, 0].max() + 0.5
    y_min, y_max = X_all[:, 1].min() - 0.5, X_all[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    bg_cmap = ListedColormap(["#AEC6E8", "#F5C8A0"])
    pt_cmap = ListedColormap(COLORS)

    ax.contourf(xx, yy, Z, cmap=bg_cmap, alpha=0.6)
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=pt_cmap,
               edgecolors="k", linewidths=0.3, s=25, label="Train")
    if X_test is not None:
        ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=pt_cmap,
                   edgecolors="k", linewidths=0.3, s=60, marker="^",
                   label="Test")
        ax.legend(fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

In [ ]:
CONFIGS = [
    {
        "name":  "Small (5,)",
        "model": MLPClassifier(hidden_layer_sizes=(5,), max_iter=1000, random_state=RANDOM_STATE),
    },
    {
        "name":  "Large (100,100,100)",
        "model": MLPClassifier(hidden_layer_sizes=(100, 100, 100), max_iter=1000, alpha=0.0, random_state=RANDOM_STATE),
    },
    {
        "name":  "Large + Reg (alpha=1.0)",
        "model": MLPClassifier(hidden_layer_sizes=(100, 100, 100), max_iter=1000, alpha=1.0, random_state=RANDOM_STATE),
    },
]

for cfg in CONFIGS:
    cfg["model"].fit(X_train, y_train)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, cfg in zip(axes, CONFIGS):
    plot_decision_boundary(ax, cfg["model"], X_train, y_train, cfg["name"], X_test=X_test, y_test=y_test)
plt.suptitle("Decision boundaries on training data", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### What to Observe

- Small (5,): The boundary is smooth but misses the crescent shape — the model *underfits* because it lacks capacity.
- Large (100,100,100): The boundary is jagged and wraps tightly around individual training points — this is *overfitting*. The model memorises noise.
- Large + Reg (alpha=1.0): The boundary shows a better shape without excessive wiggling. L2 regularization penalises large weights, smoothing the boundary.

> **Key insight:** More parameters does not equal a better model.

In [ ]:
rows = []
for cfg in CONFIGS:
    train_acc = accuracy_score(y_train, cfg["model"].predict(X_train))
    test_acc  = accuracy_score(y_test,  cfg["model"].predict(X_test))
    rows.append({
        "Model":          cfg["name"],
        "Train Accuracy": f"{train_acc:.3f}",
        "Test Accuracy":  f"{test_acc:.3f}",
        "Train - Test":   f"{train_acc - test_acc:+.3f}",
    })
pd.DataFrame(rows).set_index("Model")

## Part 2: How Do We Choose Hyperparameters Systematically?

In Part 1 we chose `hidden_layer_sizes=(100,100,100)` and `alpha=1.0` arbitrarily. We got lucky. However, in practice:

- Maybe `(50,)` is enough capacity and trains faster.
- Maybe `alpha=1.0` is too weak or strong.
- Maybe `tanh` activations work better than `relu` on this data.

**GridSearchCV** automates this search. It trains a model for every combination of hyperparameters and evaluates each with *k*-fold cross-validation **on the training set only**, returning the combination with the highest mean validation accuracy.

> We never touch the test set during the search — it is reserved for the final unbiased evaluation.

In [ ]:
param_grid = {
    "hidden_layer_sizes": [(5,), (50,50,50), (100, 100,100)],
    "alpha":              [0.01, 0.1, 1.0, 10.0],
    "activation":         ["relu", "tanh"],
}

gs = GridSearchCV(
    MLPClassifier(max_iter=1000, random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
gs.fit(X_train, y_train)

In [ ]:
print("Best hyperparameters:", gs.best_params_)
print(f"Best CV accuracy:     {gs.best_score_:.4f}")

best_model = gs.best_estimator_
test_acc = accuracy_score(y_test, best_model.predict(X_test))
print(f"Test accuracy:        {test_acc:.4f}")

In [ ]:
cm = confusion_matrix(y_test, best_model.predict(X_test))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=["Class 0", "Class 1"])
fig, ax = plt.subplots(figsize=(4, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — Best Model (Test Set)")
plt.tight_layout()
plt.show()

## Summary

### Architecture and Regularization Shape the Boundary

A network that is too small *underfits*: it cannot represent the true decision boundary regardless of training time. A network that is too large *overfits*: it fits training noise and generalises poorly. L2 regularization (`alpha`) penalises large weights and acts as a smoothness prior, recovering good generalisation at high capacity.

### Hyperparameter Search Should Be Systematic

Manually tuning hyperparameters is inefficient and hard to reproduce. `GridSearchCV` with cross-validation gives us a principled, reproducible way to search the hyperparameter space while using only the training data. The test set remains a clean, unbiased estimate of real-world performance.